In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"

DRIVE_OFFICIAL_REPO = f"{DRIVE_ROOT}/official_structured_initialization"
DRIVE_RESULTS_BACKUP = f"{DRIVE_ROOT}/official_structured_results"

LOCAL_REPO = "/content/official_structured_initialization"
LOCAL_CODE = f"{LOCAL_REPO}/pytorch-image-models-1.0.22"
LOCAL_DATA = "/content/official_structured_data"
LOCAL_RESULTS = "/content/official_structured_results"

!mkdir -p "$LOCAL_DATA" "$LOCAL_RESULTS" "$DRIVE_RESULTS_BACKUP"
!nvidia-smi

In [ ]:
%cd "$DRIVE_ROOT"

import os

if not os.path.exists(DRIVE_OFFICIAL_REPO):
    !git clone https://github.com/osiriszjq/structured_initialization.git "$DRIVE_OFFICIAL_REPO"

!rm -rf "$LOCAL_REPO"
!cp -r "$DRIVE_OFFICIAL_REPO" "$LOCAL_REPO"

%cd "$LOCAL_CODE"
!pip install -q -r requirements.txt
!pip install -q -e .


In [ ]:
import subprocess
import sys
import time

def run_official_fast(
    dataset="CIFAR10",
    num_classes=10,
    init="default",
    seed=0,
    epochs=100,
    warmup_epochs=10,
    batch_size=512,
    workers=8,
):
    exp = f"{dataset.lower()}_official_{init}_seed{seed}_{epochs}ep_bs{batch_size}"

    cmd = [
        "python", "-u", "train.py", LOCAL_DATA,
        "--dataset", f"torch/{dataset}",
        "--dataset-download",
        "--input-size", "3", "224", "224",
        "--mean", "0.485", "0.456", "0.406",
        "--std", "0.229", "0.224", "0.225",
        "--seed", str(seed),
        "--num-classes", str(num_classes),
        "--model", "vit_tiny_patch16_224",
        "--model-kwargs",
        "img_size=224",
        "weight_init=skip",
        f"post_weight_init={init}",
        "--model-ema-decay", "0.9999",
        "-j", str(workers),
        "-b", str(batch_size),
        "--lr", "2e-3",
        "--layer-decay", "1.0",
        "--warmup-lr", "1e-6",
        "--min-lr", "1e-6",
        "--weight-decay", "0.05",
        "--opt", "adamw",
        "--opt-eps", "1e-8",
        "--epochs", str(epochs),
        "--sched", "cosine",
        "--warmup-epochs", str(warmup_epochs),
        "--amp",
        "--aa", "rand-m9-mstd0.5-inc1",
        "--cutmix", "1.0",
        "--mixup", "0.8",
        "--reprob", "0.25",
        "--smoothing", "0.1",
        "--drop", "0.0",
        "--color-jitter", "0.4",
        "--drop-path", "0.1",
        "--crop-pct", "0.875",
        "--pin-mem",

        # Reduce checkpoint clutter.
        "--checkpoint-hist", "1",
        "--recovery-interval", "0",

        "--output", LOCAL_RESULTS,
        "--experiment", exp,
    ]

    print("\n" + "=" * 90)
    print(f"Starting experiment: {exp}")
    print(f"Dataset={dataset}, init={init}, seed={seed}, epochs={epochs}, batch_size={batch_size}")
    print("Command:")
    print(" ".join(cmd))
    print("=" * 90 + "\n", flush=True)

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    last_progress_time = time.time()
    recent_lines = []

    for line in process.stdout:
        line = line.rstrip()
        recent_lines.append(line)
        recent_lines = recent_lines[-8:]

        # Print important lines immediately.
        important = (
            "Train:" in line
            or "Test:" in line
            or "Epoch:" in line
            or "Acc@" in line
            or "loss" in line.lower()
            or "error" in line.lower()
            or "warning" in line.lower()
        )

        if important:
            print(line, flush=True)

        # Heartbeat every 60 seconds, useful when timm is quiet.
        if time.time() - last_progress_time > 60:
            print("\n--- still running; recent output ---")
            for l in recent_lines:
                print(l)
            print("-----------------------------------\n", flush=True)
            last_progress_time = time.time()

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(f"{exp} failed with exit code {return_code}")

    print(f"\nFinished experiment: {exp}\n", flush=True)

In [ ]:
%cd "$LOCAL_CODE"

for seed in [1, 2]:
    for init in ["default", "mimetic", "impulse"]:
        run_official_fast(
            dataset="CIFAR10",
            num_classes=10,
            init=init,
            seed=seed,
            epochs=250,
            warmup_epochs=10,
            batch_size=1024,
            workers=8,
        )

In [ ]:
!mkdir -p "$DRIVE_RESULTS_BACKUP"
!cp -r "$LOCAL_RESULTS"/* "$DRIVE_RESULTS_BACKUP"/
print("Backed up results to:", DRIVE_RESULTS_BACKUP)